In [1]:
import os
import pandas as pd
import numpy as np

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
# Set up paths
SCRIPT_DIR_PATH = os.getcwd()
CW_DIR_PATH = os.path.join(SCRIPT_DIR_PATH, "cw")
DATA_DIR_PATH = os.path.join(SCRIPT_DIR_PATH, "data")
ENSEMBLE_DATA_DIR_PATH = os.path.join(DATA_DIR_PATH, "ensemble_data")

In [4]:
# Output folders
ensemble_id = "2025-08-28t15;29;22.344855" #NOTE: Change this to your ensemble ID
RUN_ENSEMBLE_DIR_PATH = os.path.join(ENSEMBLE_DATA_DIR_PATH, f"sisepuede_summary_results_run_sisepuede_run_{ensemble_id}")
DECOMPOSED_FILE_PATH = os.path.join(RUN_ENSEMBLE_DIR_PATH, "sisepuede_results_IDE_2025-08-28t15;29;22.344855.csv")

In [5]:
decomp_ssp_df = pd.read_csv(DECOMPOSED_FILE_PATH)
decomp_ssp_df.head()

,primary_id,region,time_period,area_agrc_crops_bevs_and_spices,area_agrc_crops_cereals,area_agrc_crops_fibers,area_agrc_crops_fruits,area_agrc_crops_herbs_and_other_perennial_crops,area_agrc_crops_nuts,area_agrc_crops_other_annual,...,emission_co2e_subsector_total_lsmm,emission_co2e_subsector_total_lvst,emission_co2e_subsector_total_scoe,emission_co2e_subsector_total_soil,emission_co2e_subsector_total_trns,emission_co2e_subsector_total_trww,emission_co2e_subsector_total_waso,design_id,strategy_id,future_id
0,332332,louisiana,7,0,356696.043492,66146.621297,77.773211,76769.151304,6508.599365,1.119173e+06,...,0.230319,1.539811,4.491483,1.233181,45.130223,0.447204,3.138402,4,0,0
1,332332,louisiana,8,0,355221.860914,65873.245131,77.451784,76451.873477,6481.700093,1.114548e+06,...,0.229389,1.516623,4.525427,1.226921,45.865290,0.454013,3.189568,4,0,0
2,332332,louisiana,9,0,353750.075712,65600.313541,77.130879,76135.111621,6454.844566,1.109930e+06,...,0.228545,1.493794,4.561015,1.210181,46.698636,0.461144,3.239706,4,0,0
3,332332,louisiana,10,0,352280.764654,65327.840762,76.810514,75818.882257,6428.034184,1.105320e+06,...,0.227765,1.471320,4.598400,1.181498,47.611029,0.468528,3.291336,4,0,0
4,332332,louisiana,11,0,350814.002751,65055.840705,76.490704,75503.201530,6401.270317,1.100718e+06,...,0.227044,1.449198,4.637649,1.140642,48.587463,0.476114,3.343789,4,0,0


In [6]:
# Check unique values in strategy_id
decomp_ssp_df.strategy_id.unique()

array([   0, 6004, 6005, 6006])

In [7]:
# Filter to only include specific strategy_id
strategy_ids_to_keep = [0, 6004]
decomp_ssp_df = decomp_ssp_df[decomp_ssp_df.strategy_id.isin(strategy_ids_to_keep)]
decomp_ssp_df.strategy_id.unique()

array([   0, 6004])

In [8]:
# check how many unique future ids we have
decomp_ssp_df.future_id.nunique()

991

In [9]:
# check how many unique design ids we have
decomp_ssp_df.design_id.nunique()

1

In [10]:
decomp_ssp_df.primary_id.nunique()

992

In [12]:
from itertools import combinations

# Find pairs of primary_id that share the same future_id

# Group by future_id and collect primary_ids for each future_id
future_id_to_primary_ids = decomp_ssp_df.groupby('future_id')['primary_id'].unique()

# Create a list of tuples (future_id, primary_id1, primary_id2) for pairs sharing the same future_id
shared_future_id_pairs = []
for future_id, primary_ids in future_id_to_primary_ids.items():
    if len(primary_ids) > 1:
        for pid1, pid2 in combinations(primary_ids, 2):
            shared_future_id_pairs.append((future_id, pid1, pid2))

# Display the result
shared_future_id_pairs[:10]  # Show first 10 pairs

[(0, np.int64(332332), np.int64(402402))]

## future_id 0 is duplicated

In [13]:
# Drop unnecessary columns
columns_to_drop = ["strategy_id", "design_id", "future_id"]
decomp_ssp_df = decomp_ssp_df.drop(columns=columns_to_drop)
decomp_ssp_df.head()

,primary_id,region,time_period,area_agrc_crops_bevs_and_spices,area_agrc_crops_cereals,area_agrc_crops_fibers,area_agrc_crops_fruits,area_agrc_crops_herbs_and_other_perennial_crops,area_agrc_crops_nuts,area_agrc_crops_other_annual,...,emission_co2e_subsector_total_inen,emission_co2e_subsector_total_ippu,emission_co2e_subsector_total_lndu,emission_co2e_subsector_total_lsmm,emission_co2e_subsector_total_lvst,emission_co2e_subsector_total_scoe,emission_co2e_subsector_total_soil,emission_co2e_subsector_total_trns,emission_co2e_subsector_total_trww,emission_co2e_subsector_total_waso
0,332332,louisiana,7,0,356696.043492,66146.621297,77.773211,76769.151304,6508.599365,1.119173e+06,...,117.042622,2.762559,-0.077001,0.230319,1.539811,4.491483,1.233181,45.130223,0.447204,3.138402
1,332332,louisiana,8,0,355221.860914,65873.245131,77.451784,76451.873477,6481.700093,1.114548e+06,...,160.204394,2.773548,-0.104854,0.229389,1.516623,4.525427,1.226921,45.865290,0.454013,3.189568
2,332332,louisiana,9,0,353750.075712,65600.313541,77.130879,76135.111621,6454.844566,1.109930e+06,...,116.085554,2.786645,-0.132605,0.228545,1.493794,4.561015,1.210181,46.698636,0.461144,3.239706
3,332332,louisiana,10,0,352280.764654,65327.840762,76.810514,75818.882257,6428.034184,1.105320e+06,...,163.932900,2.801782,-0.160253,0.227765,1.471320,4.598400,1.181498,47.611029,0.468528,3.291336
4,332332,louisiana,11,0,350814.002751,65055.840705,76.490704,75503.201530,6401.270317,1.100718e+06,...,163.431885,2.818877,-0.187795,0.227044,1.449198,4.637649,1.140642,48.587463,0.476114,3.343789


In [14]:
decomp_ssp_df.to_csv(os.path.join(RUN_ENSEMBLE_DIR_PATH, "sisepuede_results_IDE_6004_filtered.csv"), index=False)

## Filter the Attribute Primary table

In [15]:
attr_primary_df = pd.read_csv(os.path.join(RUN_ENSEMBLE_DIR_PATH, "ATTRIBUTE_PRIMARY.csv"))
attr_primary_df.head()

,primary_id,design_id,strategy_id,future_id
0,332332,4,0,0
1,402402,4,6004,0
2,402403,4,6004,1
3,402404,4,6004,2
4,402405,4,6004,3


In [16]:
# check unique strategy_ids
attr_primary_df.strategy_id.unique()

array([   0, 6004, 6005, 6006])

In [17]:
# Filter to only include the strategy_ids we want
attr_primary_df = attr_primary_df[attr_primary_df.strategy_id.isin(strategy_ids_to_keep)]
attr_primary_df.strategy_id.unique()

array([   0, 6004])

In [18]:
# Check design id
attr_primary_df.design_id.nunique()

1

In [19]:
# Check nunique future id
attr_primary_df.future_id.nunique()

1001

In [20]:
# We need to force a match between the filtered decomp_ssp_df and attr_primary_df primary ids
valid_primary_ids = decomp_ssp_df.primary_id.unique()
attr_primary_df = attr_primary_df[attr_primary_df.primary_id.isin(valid_primary_ids)]
attr_primary_df.primary_id.nunique()

992

In [21]:
attr_primary_df.head()

,primary_id,design_id,strategy_id,future_id
0,332332,4,0,0
1,402402,4,6004,0
2,402403,4,6004,1
3,402404,4,6004,2
4,402405,4,6004,3


In [22]:
# re-compute future_id
attr_primary_df["new_future_id"] = attr_primary_df["future_id"] + 1
attr_primary_df.head()

,primary_id,design_id,strategy_id,future_id,new_future_id
0,332332,4,0,0,1
1,402402,4,6004,0,1
2,402403,4,6004,1,2
3,402404,4,6004,2,3
4,402405,4,6004,3,4


In [23]:
# drop future id and rename new_future_id to future_id
attr_primary_df = attr_primary_df.drop(columns=["future_id"])
attr_primary_df = attr_primary_df.rename(columns={"new_future_id": "future_id"})
attr_primary_df.head()

,primary_id,design_id,strategy_id,future_id
0,332332,4,0,1
1,402402,4,6004,1
2,402403,4,6004,2
3,402404,4,6004,3
4,402405,4,6004,4


In [24]:
# save filtered ATTRIBUTE_PRIMARY.csv
attr_primary_df.to_csv(os.path.join(RUN_ENSEMBLE_DIR_PATH, "ATTRIBUTE_PRIMARY_6004_filtered.csv"), index=False)